# Experimentación con canary 🐤

In [3]:
import torch

print(f"Is CUDA available? {torch.cuda.is_available()}")

gpu_count = torch.cuda.device_count()
print(f"Number of GPUs detected: {gpu_count}")

for i in range(gpu_count):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

Is CUDA available? True
Number of GPUs detected: 2
GPU 0: NVIDIA RTX A5000
GPU 1: NVIDIA RTX A5000


## Modelo

In [1]:
import nemo.collections.asr as nemo_asr

[NeMo W 2026-03-13 15:25:16 nemo_logging:364] /home/umoqnier/iwstl-low-resource-experiments/.venv/lib/python3.13/site-packages/lhotse/recipes/iwslt22_ta.py:323: SyntaxWarning: invalid escape sequence '\s'
      text = re.sub("\s+", " ", text)
    
[NeMo W 2026-03-13 15:25:16 nemo_logging:364] /home/umoqnier/iwstl-low-resource-experiments/.venv/lib/python3.13/site-packages/lhotse/recipes/iwslt22_ta.py:324: SyntaxWarning: invalid escape sequence '\s'
      text = re.sub("\s+\.\s+", ".", text)
    
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-03-13 15:25:17 nemo_logging:364] /home/umoqnier/iwstl-low-resource-experiments/.venv/lib/python3.13/site-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
      m = re.match('([su]([0-9]{1,2

In [48]:
model = nemo_asr.models.ASRModel.from_pretrained("nvidia/canary-1b-v2")

canary-1b-v2.nemo:   0%|          | 0.00/6.36G [00:00<?, ?B/s]

[NeMo I 2026-03-13 16:08:33 mixins:184] Tokenizer CanaryBPETokenizer initialized with 16384 tokens


[NeMo W 2026-03-13 16:08:33 modelPT:176] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 4
    pin_memory: true
    prompt_format: canary2
    max_duration: 40.0
    min_duration: 0.01
    text_field: answer
    lang_field: target_lang
    use_bucketing: true
    max_tps: null
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: null
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    
[NeMo W 2026-03-13 16:08:33 modelPT:183] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation

[NeMo I 2026-03-13 16:08:39 save_restore_connector:146] Restoration will occur within pre-extracted directory : `/tmp/tmpjfldr3e7`.
[NeMo I 2026-03-13 16:09:01 mixins:184] Tokenizer SentencePieceTokenizer initialized with 16384 tokens


[NeMo W 2026-03-13 16:09:02 modelPT:176] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 40.0
    min_duration: 0.1
    text_field: answer
    batch_duration: null
    max_tps: null
    use_bucketing: true
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: null
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    
[NeMo W 2026-03-13 16:09:02 modelPT:183] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validatio

[NeMo I 2026-03-13 16:09:05 save_restore_connector:286] Model EncDecCTCModelBPE was successfully restored from /tmp/tmpjfldr3e7.
[NeMo I 2026-03-13 16:09:06 save_restore_connector:286] Model EncDecMultiTaskModel was successfully restored from /home/umoqnier/.cache/huggingface/hub/models--nvidia--canary-1b-v2/snapshots/87bc52657add533cd0156b3fc1aef027280754bf/canary-1b-v2.nemo.


## Habilitando soporte de Adapters

In [51]:
model.replace_adapter_compatible_modules()

[NeMo I 2026-03-13 16:13:29 adapter_mixins:169] Swapping class nemo.collections.asr.modules.conformer_encoder.ConformerEncoder with adapter compatible class: nemo.collections.asr.modules.conformer_encoder.ConformerEncoderAdapter
[NeMo I 2026-03-13 16:13:29 adapter_mixins:169] Swapping class nemo.collections.asr.modules.transformer.transformer.TransformerDecoderNM with adapter compatible class: nemo.collections.asr.modules.transformer.transformer.TransformerDecoderNMAdapter
[NeMo I 2026-03-13 16:13:29 adapter_mixins:169] Swapping class nemo.collections.asr.modules.transformer.transformer_decoders.TransformerDecoder with adapter compatible class: nemo.collections.asr.modules.transformer.transformer_decoders.TransformerDecoderAdapter


### Verificación de que *targets* son soportados por el modelo

In [52]:
model.adapter_module_names

['', 'encoder', 'transf_encoder', 'transf_decoder']

### Preparando los *Adapters*

In [53]:
from nemo.collections.common.parts import LinearAdapterConfig

In [54]:
input_dim = model.cfg.encoder.d_model
adapter_dim = 8

In [55]:
enc_adapter_cfg = LinearAdapterConfig(in_features=input_dim, dim=adapter_dim)
dec_adapter_cfg = LinearAdapterConfig(in_features=input_dim, dim=adapter_dim)

### Agregando *Adapters*

In [56]:
model.add_adapter(name="encoder:enc", cfg=enc_adapter_cfg)
model.add_adapter(name="transf_decoder:dec", cfg=dec_adapter_cfg)

### Congelando parámetros del modelo y descongelando solo los pesos de *Adapters*

In [57]:
model.freeze()
model.unfreeze_enabled_adapters()

[NeMo I 2026-03-13 16:13:35 adapter_mixins:466] Froze module encoder.layers.0.conv.batch_norm: BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
[NeMo I 2026-03-13 16:13:35 adapter_mixins:466] Froze module encoder.layers.1.conv.batch_norm: BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
[NeMo I 2026-03-13 16:13:35 adapter_mixins:466] Froze module encoder.layers.2.conv.batch_norm: BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
[NeMo I 2026-03-13 16:13:35 adapter_mixins:466] Froze module encoder.layers.3.conv.batch_norm: BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
[NeMo I 2026-03-13 16:13:35 adapter_mixins:466] Froze module encoder.layers.4.conv.batch_norm: BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
[NeMo I 2026-03-13 16:13:35 adapter_mixins:466] Froze module encoder.layers.5.conv.batch_norm: BatchNorm1d(102

In [58]:
model.summarize()

  | Name                 | Type                              | Params | Mode
----------------------------------------------------------------------------------
0 | preprocessor         | AudioToMelSpectrogramPreprocessor | 0      | eval
1 | encoder              | ConformerEncoderAdapter           | 811 M  | eval
2 | encoder_decoder_proj | Identity                          | 0      | eval
3 | transf_decoder       | TransformerDecoderNMAdapter       | 151 M  | eval
4 | log_softmax          | TokenClassifier                   | 16.8 M | eval
5 | loss                 | SmoothedCrossEntropyLoss          | 0      | eval
6 | spec_augmentation    | SpectrogramAugmentation           | 0      | eval
7 | val_loss             | GlobalAverageLossMetric           | 0      | eval
8 | wer                  | WER                               | 0      | eval
9 | bleu                 | BLEU                              | 0      | eval
----------------------------------------------------------------------

### Comprobando los *Adapters habilidatos*

In [59]:
model.get_enabled_adapters()

['dec', 'enc']

## Dataset - Mapuche

In [60]:
from huggingface_hub import login

login()

In [ ]:
# Do it on terminal
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 

In [61]:
!hf auth whoami

user:  umoqnier
orgs:  ElotlMX,somosnlp-hackathon-2022,somosnlp,UNAMMexico


In [13]:
from datasets import load_dataset, load_dataset_builder

### Exploración del dataset

In [14]:
MAP_DATASET_ID = "mengct00/Mapudungun_iwslt26"

In [15]:
map_databuilder = load_dataset_builder(MAP_DATASET_ID)

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/121 [00:00<?, ?it/s]

In [16]:
map_databuilder.info.features

{'audio': Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None),
 'arn': Value('string'),
 'arn_clean': Value('string'),
 'spa': Value('string')}

In [17]:
map_databuilder.info.splits

{'train': SplitInfo(name='train', num_bytes=24941130380, num_examples=41092, shard_lengths=None, original_shard_lengths=None, dataset_name=None),
 'validation': SplitInfo(name='validation', num_bytes=774427821, num_examples=1201, shard_lengths=None, original_shard_lengths=None, dataset_name=None)}

### Preparación del dataset

Canary hace uso de un *prompt* que maneja que tipo de tarea vamos a resolver. Necesitamos entregarle los datos en un formato que cumpla con el formato de la clase que define el *prompt*.

In [18]:
model.prompt?

Type:            Canary2PromptFormatter
String form:     <nemo.collections.common.prompts.canary2.Canary2PromptFormatter object at 0x7ebe4fe6acf0>
File:            ~/iwstl-low-resource-experiments/.venv/lib/python3.13/site-packages/nemo/collections/common/prompts/canary2.py
Docstring:       <no docstring>
Class docstring:
:class:`~nemo.collections.common.prompts.formatter.PromptFormatter` is intended to simplify
working with various prompt format templates and encoding them into token ID tensors.

It assumes a dialog-like structure, which is a list of turns, with each turn assigned to a role.
Sub-classes of PromptFormatter define turn templates for each role under TEMPLATE class attribute.
Each template may define some constant parts (e.g. begin-of-turn or end-of-turn tokens, whitespaces, etc.)
and variable parts which we call "slots", that will be provided by the user during training or inference.

A role is typically "user" and "assistant", and some popular models also use a "system"

In [19]:
model.prompt.TEMPLATE

{'user': {'template': '<|startofcontext|>|decodercontext|<|startoftranscript|>|emotion||source_lang||target_lang||pnc||itn||timestamp||diarize|',
  'slots': {'decodercontext': nemo.collections.common.prompts.formatter.Text,
   'emotion': Modality.TextLiteral(allowed_values=('<|emo:undefined|>', '<|emo:neutral|>', '<|emo:angry|>', '<|emo:happy|>', '<|emo:sad|>')),
   'source_lang': nemo.collections.common.prompts.formatter.Text,
   'target_lang': nemo.collections.common.prompts.formatter.Text,
   'pnc': Modality.TextLiteral(allowed_values=('1', 'False', 'pnc', '<|nopnc|>', 'no', 'Yes', '0', 'True', 'yes', 'true', 'nopnc', 'false', '<|pnc|>', 'No')),
   'itn': Modality.TextLiteral(allowed_values=('1', 'False', 'no', 'Yes', '0', '<|noitn|>', 'itn', 'noitn', 'True', 'yes', 'true', 'false', '<|itn|>', 'No')),
   'timestamp': Modality.TextLiteral(allowed_values=('1', 'False', 'no', 'Yes', '0', '<|timestamp|>', '<|notimestamp|>', 'True', 'yes', 'true', 'notimestamp', 'timestamp', 'false', 'No

Usaremos la clase `PromptedAudioToTextLhotseDataset` predefinida en la biblioteca de Nemo. Esta clase mapea items del manifest definido por nosotros  a items definidos en el *prompt template* del modelo. Asi, mientras el *manifest* corresponda con los *slots* soportados por el modelo, estos seran manejados por el Dataset automáticamente. 

In [20]:
import torch
from nemo.collections.asr.data.audio_to_text_lhotse_prompted import (
    PromptedAudioToTextMiniBatch,
)
from nemo.collections.common.data.prompt_fn import (
    get_prompt_format_fn,
    #registered_prompt_format_fn,
)
from torch.utils.data import Dataset
from lhotse.dataset import AudioSamples
from lhotse.dataset.collation import collate_vectors
from lhotse import CutSet
from nemo.collections.common.prompts import PromptFormatter
from lhotse.cut import Cut


class MyCanaryPromptedAudioToTextLhotseDataset(Dataset):
    """
    This dataset is based on :class:`~nemo.collections.asr.data.audio_to_text_lhotse.LhotseSpeechToTextBpeDataset`.
    It is a Lhotse-style dataset that converts a mini-batch of Cuts into tensors.
    The main difference from ``LhotseSpeechToTextBpeDataset`` is that we introduce
    a special prompt format for multitask encoder-decoder models.

    To perform the prompt formatting, we accept a ``prompt_format_fn``.
    It's expected to accept:
    * a ``Cut`` a single MonoCut or MixedCut
    * a ``PromptFormatter`` Prepend and append control tokens to the token sequence

    Tokenized utterances will be extended with special prompt tokens according to ``prompt_format_fn`` logic.
    We support cuts with multiple supervision segments -- their tokenized texts will be concatenated before we add the prompt tokens.
    This is useful, for example, in code-switched scenarios where each segment is spoken in a different language.
    """

    def __init__(self, tokenizer: "TokenizerSpec", prompt: PromptFormatter):
        super().__init__()
        self.tokenizer = tokenizer
        self.load_audio = AudioSamples(fault_tolerant=True)
        self.padding_value = self.tokenizer.pad_id
        self.prompt = prompt
        self.prompt_format_fn = get_prompt_format_fn(
            Cut, self.prompt
        )  # Use the default canary prompt function

    def __getitem__(self, cuts: CutSet) -> PromptedAudioToTextMiniBatch:
        audio, audio_lens, cuts = self.load_audio(cuts)
        answers = []
        prompts = []
        prompts_with_answers = []

        for cut in cuts:
            prompted_answers = self.prompt_format_fn(cut, self.prompt)
            answers.append(prompted_answers["answer_ids"])
            prompts.append(prompted_answers["context_ids"])
            prompts_with_answers.append(prompted_answers["input_ids"])

        transcript, transcript_lens = self._collate_tokens(answers)
        prompts_with_answers, prompts_with_answers_lens = self._collate_tokens(
            prompts_with_answers
        )
        prompts, prompt_lens = self._collate_tokens(prompts)

        return PromptedAudioToTextMiniBatch(
            audio=audio,
            audio_lens=audio_lens,
            transcript=transcript,
            transcript_lens=transcript_lens,
            prompt=prompts,
            prompt_lens=prompt_lens,
            prompted_transcript=prompts_with_answers,
            prompted_transcript_lens=prompts_with_answers_lens,
            cuts=cuts.drop_in_memory_data(),
        )

    def _collate_tokens(
        self, tokens: list[list[int] | torch.Tensor]
    ) -> tuple[torch.Tensor, torch.Tensor]:
        tokens = [torch.as_tensor(t) for t in tokens]
        token_lens = torch.tensor([t.size(0) for t in tokens], dtype=torch.long)
        tokens = collate_vectors(tokens, padding_value=self.padding_value)
        return tokens, token_lens


In [21]:
import os
import tqdm
import soundfile as sf
import lightning.pytorch as L
from omegaconf import OmegaConf
from nemo.collections.asr.parts.utils.manifest_utils import write_manifest, read_manifest
from nemo.collections.common.data.lhotse import get_lhotse_dataloader_from_config
from nemo.collections.common.prompts import CanaryPromptFormatter

In [22]:
class CanaryMapudungunDataModule(L.LightningDataModule):
    def __init__(
        self, 
        tokenizer, 
        data_dir: str = "./map_data/", 
        batch_size=8,
        dataset_name=MAP_DATASET_ID,
        streaming: bool = False,
        max_samples: int = None,
    ):
        super().__init__()
        self.tokenizer = tokenizer
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.dataset_name = dataset_name
        self.streaming = streaming
        self.max_samples = max_samples

        # AST manifests paths
        self.train_manifest = os.path.join(data_dir, "train_manifest.json")
        self.val_manifest = os.path.join(data_dir, "val_manifest.json")
        #self.test_manifest = os.path.join(data_dir, "test_manifest.json")

    def setup(self, stage=None):
        pass

    def _setup_dataloader(self, config):
        """
        The main function that creates the data loader using Lhotse's integration with NeMo.
        """
        # Ensure trainer is attached before accessing rank
        rank = self.trainer.global_rank if self.trainer else 0
        world_size = self.trainer.world_size if self.trainer else 1
        
        return get_lhotse_dataloader_from_config(
            OmegaConf.create(config),
            global_rank=rank,
            world_size=world_size,
            dataset=MyCanaryPromptedAudioToTextLhotseDataset(
                tokenizer=self.tokenizer, prompt=CanaryPromptFormatter(self.tokenizer)
            ),
    )
    def train_dataloader(self):
        config = {
            "manifest_filepath": self.train_manifest,
            "batch_size": self.batch_size,
            "num_workers": 4,
            "shuffle": True,
            "min_duration": 0.1,
            "max_duration": 20.0,
        }
        return self._setup_dataloader(config)

    def val_dataloader(self):
        config = {
            "manifest_filepath": self.val_manifest,
            "batch_size": self.batch_size,
            "num_workers": 4,
            "shuffle": False,
            "min_duration": 0.1,
            "max_duration": 20.0,
        }
        return self._setup_dataloader(config)

    def test_dataloader(self):
        config = {
            "manifest_filepath": self.test_manifest,
            "batch_size": self.batch_size,
            "num_workers": 4,
            "shuffle": False,
            "min_duration": 0.1,
            "max_duration": 20.0,
        }
        return self._setup_dataloader(config)
    
    def prepare_data(self):
        """
        Downloads the HF dataset, saves audio to disk, and creates NeMo manifests.
        Only called on 1 GPU/TPU in distributed settings.
        """
        if not os.path.exists(self.data_dir):
            os.makedirs(self.data_dir)

        # Force re-creation if max_samples is set (prototyping mode)
        if not self.max_samples and os.path.exists(self.train_manifest) and os.path.exists(self.val_manifest):
            print("Manifests found. Skipping data preparation.")
            return

        print(f"Downloading and processing dataset: {self.dataset_name} (streaming={self.streaming})")
        dataset = load_dataset(self.dataset_name, streaming=self.streaming)
        
        # Audio directory to store extracted .wav files
        wav_dir = os.path.join(self.data_dir, "wavs")
        os.makedirs(wav_dir, exist_ok=True)

        # Define mapping from HF dataset splits to manifest paths
        available_splits = dataset.keys()
        split_mapping = {}
        if "train" in available_splits:
            split_mapping["train"] = self.train_manifest
        if "validation" in available_splits:
            split_mapping["validation"] = self.val_manifest
        if "test" in available_splits:
            split_mapping["test"] = self.test_manifest

        for split, manifest_path in split_mapping.items():
            print(f"Processing split: {split}...")
            data = dataset[split]
            
            # Apply limit if prototyping
            if self.max_samples is not None:
                print(f"Limiting to {self.max_samples} samples for split '{split}'")
                if self.streaming:
                    data = data.take(self.max_samples)
                else:
                    data = data.select(range(min(len(data), self.max_samples)))

            manifest_entries = []
            
            # For streaming, we can't always know total length, handling tqdm accordingly
            try:
                total = len(data)
            except TypeError:
                total = self.max_samples

            for idx, item in tqdm.tqdm(enumerate(data), total=total, desc=f"Processing {split}"):
                # 1. Handle Audio
                audio_info = item.get("audio")
                if not audio_info:
                    print(f"Warning: No audio found for item {idx} in {split}")
                    continue
                
                # Construct a unique filename for the audio
                file_id = f"{split}_{idx}"
                filename = f"{file_id}.wav"
                filepath = os.path.join(wav_dir, filename)
                
                # Save audio file to disk
                # Even if file exists, we might want to overwrite if prototyping with different sampling
                # or ensure integrity.
                sf.write(filepath, audio_info["array"], audio_info["sampling_rate"])
                
                duration = len(audio_info["array"]) / audio_info["sampling_rate"]

                # 2. Handle Text (Target Translation)
                text = item.get("spa", "")
                
                if not text:
                    for k, v in item.items():
                        if isinstance(k, str) and (k.endswith("_es") or k.endswith("_spa")) and isinstance(v, str):
                            text = v
                            break
                
                if not text:
                    print(f"Warning: No text found for item {idx} in {split}")
                    continue

                # 3. Create manifest entry
                entry = {
                    "audio_filepath": os.path.abspath(filepath),
                    "duration": duration,
                    "text": text,
                    "pnc": "yes",
                    "task": "ast",           # Task is AST
                    "source_lang": "arn",    # Mapudungun ISO code
                    "target_lang": "es",     # Spanish ISO code
                }
                manifest_entries.append(entry)

            write_manifest(manifest_path, manifest_entries)
            print(f"Created manifest: {manifest_path}")

        print("Data preparation complete.")

In [23]:
data_loader = CanaryMapudungunDataModule(tokenizer=model.tokenizer, streaming=True, max_samples=100, batch_size=16)

In [24]:
data_loader.prepare_data()

Resolving data files:   0%|          | 0/121 [00:00<?, ?it/s]

Processing split: train...
Limiting to 100 samples for split 'train'


Processing train: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:19<00:00,  5.23it/s]


Created manifest: ./map_data/train_manifest.json
Processing split: validation...
Limiting to 100 samples for split 'validation'


Processing validation: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:18<00:00,  5.55it/s]

Created manifest: ./map_data/val_manifest.json
Data preparation complete.


In [25]:
!head -n 5 {data_loader.train_manifest}

{"audio_filepath": "/home/umoqnier/iwstl-low-resource-experiments/map_data/wavs/train_0.wav", "duration": 4.172335600907029, "text": "Hola hermano, \u00bfc\u00f3mo te llamas t\u00fa?", "pnc": "yes", "task": "ast", "source_lang": "arn", "target_lang": "es"}
{"audio_filepath": "/home/umoqnier/iwstl-low-resource-experiments/map_data/wavs/train_1.wav", "duration": 30.20408163265306, "text": "Hola hermana. Mi nombre Agust\u00edn Curiqueo Quintriqueo se dice mi nombre. Mi tierra se llama Cancura. All\u00ed est\u00e1n mi pap\u00e1, mis mam\u00e1s, mi hermano, mi hermana. El principal de mi familia se llama Jos\u00e9 Sangre Mollfunao, mi viejo es ese pues. As\u00ed es pues, hermana.", "pnc": "yes", "task": "ast", "source_lang": "arn", "target_lang": "es"}
{"audio_filepath": "/home/umoqnier/iwstl-low-resource-experiments/map_data/wavs/train_2.wav", "duration": 42.94784580498866, "text": "Ya. Yo tambi\u00e9n, mi nombre, me llamo yo soy llamada, Mar\u00eda Isabel Co\u00f1oep\u00e1n, me llamo yo. 

In [26]:
!head -n 5 {data_loader.val_manifest}

{"audio_filepath": "/home/umoqnier/iwstl-low-resource-experiments/map_data/wavs/validation_0.wav", "duration": 5.245895691609977, "text": "Pero la machi sola. sola no queda esa.", "pnc": "yes", "task": "ast", "source_lang": "arn", "target_lang": "es"}
{"audio_filepath": "/home/umoqnier/iwstl-low-resource-experiments/map_data/wavs/validation_1.wav", "duration": 15.254512471655328, "text": "Sola, sola no queda pues ya cuando la machi tiene a su ayudante que lo tomo el ayudante que lo tomo pero de arriba bajo eso padre.", "pnc": "yes", "task": "ast", "source_lang": "arn", "target_lang": "es"}
{"audio_filepath": "/home/umoqnier/iwstl-low-resource-experiments/map_data/wavs/validation_2.wav", "duration": 2.6574603174603175, "text": "En la tierra no est\u00e1 su ayudante arriba.", "pnc": "yes", "task": "ast", "source_lang": "arn", "target_lang": "es"}
{"audio_filepath": "/home/umoqnier/iwstl-low-resource-experiments/map_data/wavs/validation_3.wav", "duration": 17.46331065759637, "text": "En l

# Train Model

Ya que han sido preparados los *adapters* es tiempo de entrenar los pesos de los mismos con los datos.

Actualizamos parámetros del optimizador

In [62]:
print(OmegaConf.to_yaml(model.cfg.optim))

name: adamw
lr: 5.0e-05
betas:
- 0.9
- 0.98
weight_decay: 0.001
sched:
  name: InverseSquareRootAnnealing
  max_steps: 10000
  warmup_steps: 0
  warmup_ratio: null
  min_lr: 1.0e-06



In [63]:
# Setup optimization
model.cfg.optim.lr = 3e-4
model.cfg.optim.sched.warmup_steps = 25

Configuramos un entrenador lighting

In [64]:
trainer = L.Trainer(
    max_steps=200,
    accumulate_grad_batches=1,
    logger=False,
    enable_checkpointing=False,
    check_val_every_n_epoch=5,
)

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [65]:
trainer.fit(model, data_loader)

Resolving data files:   0%|          | 0/121 [00:00<?, ?it/s]

Processing split: train...
Limiting to 100 samples for split 'train'


Processing train: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:27<00:00,  3.60it/s]


Created manifest: ./map_data/train_manifest.json
Processing split: validation...
Limiting to 100 samples for split 'validation'


Processing validation: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:19<00:00,  5.08it/s]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Created manifest: ./map_data/val_manifest.json
Data preparation complete.
[NeMo I 2026-03-13 16:14:54 modelPT:782] Optimizer config = AdamW (
    Parameter Group 0
        amsgrad: False
        betas: (0.9, 0.98)
        capturable: False
        decoupled_weight_decay: True
        differentiable: False
        eps: 1e-08
        foreach: None
        fused: None
        lr: 0.0003
        maximize: False
        weight_decay: 0.001
    )
[NeMo I 2026-03-13 16:14:54 lr_scheduler:995] Scheduler "<nemo.core.optim.lr_scheduler.InverseSquareRootAnnealing object at 0x7ebdbd9d7e00>" 
    will be used during training (effective maximum steps = 200) - 
    Parameters : 
    (max_steps: 200
    warmup_steps: 25
    warmup_ratio: null
    min_lr: 1.0e-06
    )



  | Name                 | Type                              | Params | Mode
----------------------------------------------------------------------------------
0 | preprocessor         | AudioToMelSpectrogramPreprocessor | 0      | eval
1 | encoder              | ConformerEncoderAdapter           | 811 M  | eval
2 | encoder_decoder_proj | Identity                          | 0      | eval
3 | transf_decoder       | TransformerDecoderNMAdapter       | 151 M  | eval
4 | log_softmax          | TokenClassifier                   | 16.8 M | eval
5 | loss                 | SmoothedCrossEntropyLoss          | 0      | eval
6 | spec_augmentation    | SpectrogramAugmentation           | 0      | eval
7 | val_loss             | GlobalAverageLossMetric           | 0      | eval
8 | wer                  | WER                               | 0      | eval
9 | bleu                 | BLEU                              | 0      | eval
---------------------------------------------------------------------

Sanity Checking: |                                                                                            …

[NeMo I 2026-03-13 16:14:55 dataloader:342] We will be using a Lhotse DataLoader.
[NeMo I 2026-03-13 16:14:55 dataloader:657] Creating a Lhotse DynamicCutSampler (bucketing is disabled, (max_batch_duration=None max_batch_size=16)


KeyError: Caught KeyError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/umoqnier/iwstl-low-resource-experiments/.venv/lib/python3.13/site-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/home/umoqnier/iwstl-low-resource-experiments/.venv/lib/python3.13/site-packages/torch/utils/data/_utils/fetch.py", line 56, in fetch
    data = self.dataset[possibly_batched_index]
           ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_10350/3351560961.py", line 51, in __getitem__
    prompted_answers = self.prompt_format_fn(cut, self.prompt)
  File "/home/umoqnier/iwstl-low-resource-experiments/.venv/lib/python3.13/site-packages/nemo/collections/common/prompts/canary.py", line 187, in canary
    ans = prompt.encode_dialog(turns)
  File "/home/umoqnier/iwstl-low-resource-experiments/.venv/lib/python3.13/site-packages/nemo/collections/common/promp

In [ ]:
model.save_adapters("mapuche_adapters.pt")

In [ ]:
from torchmetrics.text import SacreBLEUScore

sacrebleu = SacreBLEUScore(n_gram=4)
scores = []
preds = []
gts = []
for pred, gt in zip(ast_preds, ast_gt):
    preds.append(pred)
    gts.append([gt])

# bleu = sum(scores) / len(scores)
sacrebleu.update([p.text for p in preds], gts)
bleu = sacrebleu.compute()
print("BLEU", bleu.item() * 100)